# Import Library

In [1]:
import os
import pandas as pd
import io
from google import genai

# Inisialisasi API KEY

In [2]:
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
client = genai.Client()

# Prompt Engineering

In [3]:
print("Meminta Gemini untuk membuat dataset sintetik... (Mohon tunggu sekitar 10-20 detik)")

prompt = """
Kamu adalah asisten pembuat dataset Machine Learning.
Buatkan saya dataset sintetik sebanyak 50 baris. 

Skenario: Laporan kerusakan aset pabrik yang berisi cerita sebab-akibat antara keluhan orang awam dan laporan perbaikan oleh teknisi.

ATURAN KETAT:
- Pisahkan antar kolom HANYA menggunakan simbol pipe (|). JANGAN gunakan koma sebagai pemisah kolom.
- Jangan berikan teks pembuka atau penutup apapun. LANGSUNG berikan data mentahnya.

KOLOM YANG WAJIB ADA (Tepat 6 kolom):
teks_keluhan_awam|teks_laporan_teknisi|kategori_aset|severity|root_cause|tindakan

Pilihan untuk 4 kolom terakhir:
- kategori_aset: (HVAC, Pompa Air, Kelistrikan, Mesin Produksi)
- severity: (Rendah, Sedang, Tinggi)
- root_cause: (Tersumbat, Keausan, Overheat, Usia Pakai, Konsleting)
- tindakan: (Pembersihan, Penggantian Part, Pelumasan, Kalibrasi, Perbaikan Jaringan)

Contoh Baris yang Benar:
AC di ruang meeting lantai 2 netes air terus nih|udah disemprot selang pembuangannya karena mampet lumut, sekalian tambah freon|HVAC|Sedang|Tersumbat|Pembersihan
"""

Meminta Gemini untuk membuat dataset sintetik... (Mohon tunggu sekitar 10-20 detik)


# Memanggil Model Gemini

In [4]:
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,
)

# Processing ke CSV

In [7]:
csv_raw_data = response.text

# 4. Membersihkan format markdown jika ada
if csv_raw_data.startswith("```"):
    # Membuang baris pertama (```text) dan baris terakhir (```)
    lines = csv_raw_data.strip().split('\n')
    csv_raw_data = '\n'.join(lines[1:-1])

# 5. Membaca data dengan pemisah | (pipe)
try:
    # PERHATIKAN: Parameter sep='|' ditambahkan di sini
    df_sintetik = pd.read_csv(io.StringIO(csv_raw_data.strip()), sep='|')
    
    # 6. Menyimpan dataset ke file CSV lokal (komputer kamu)
    file_name = "../../data/synthetic_report_dataset.csv"
    df_sintetik.to_csv(file_name, index=False)
    
    print(f"\n✅ BERHASIL! Dataset berhasil dibuat dan disimpan sebagai '{file_name}'")
    print(f"Jumlah baris: {df_sintetik.shape[0]}")
    print("\n--- 5 Baris Pertama ---")
    display(df_sintetik.head()) # Gunakan print(df_sintetik.head()) jika tidak di Jupyter
    
except Exception as e:
    print(f"❌ Gagal memproses format data. Error: {e}")
    print("Teks mentah dari Gemini:\n", csv_raw_data)


✅ BERHASIL! Dataset berhasil dibuat dan disimpan sebagai '../../data/synthetic_report_dataset.csv'
Jumlah baris: 49

--- 5 Baris Pertama ---


,AC di ruang meeting lantai 2 netes air terus nih,"Saluran pembuangan air kondensasi tersumbat lumut dan debu, sudah dibersihkan tuntas",HVAC,Sedang,Tersumbat,Pembersihan
0,"Pompa air di sumur bor suaranya kasar sekali, ...","Bearing motor pompa aus dan berkarat, sudah di...",Pompa Air,Tinggi,Keausan,Penggantian Part
1,"Mesin pengisi botol macet, produk tidak keluar...","Jalur produk tersumbat sisa bahan, sudah diber...",Mesin Produksi,Tinggi,Tersumbat,Pembersihan
2,"Lampu di lorong lantai 1 berkedip-kedip, kadan...","Konektor kabel lampu sudah aus dan longgar, su...",Kelistrikan,Rendah,Keausan,Penggantian Part
3,"AC di lab tiba-tiba mati total, ada bau gosong","Terjadi konsleting pada modul kontrol PCB, sud...",HVAC,Tinggi,Konsleting,Perbaikan Jaringan
4,Tekanan air di area produksi sangat lemah sekali,Filter inlet pompa tersumbat lumpur dan pasir ...,Pompa Air,Sedang,Tersumbat,Pembersihan
